In [ ]:
# CELL 1: Colab setup - skip if running locally in VS Code
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import subprocess, os
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', '-q'], check=True)
    repo_url = 'https://github.com/jminot92/lol-coach.git'
    if not os.path.exists('/content/lol-coach'):
        subprocess.run(['git', 'clone', repo_url, '/content/lol-coach'], check=True)
    else:
        subprocess.run(['git', '-C', '/content/lol-coach', 'pull', '--ff-only'], check=True)
    os.chdir('/content/lol-coach')
    commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
    print(f'Colab ready  |  commit: {commit}')
else:
    print('Running locally - skipping Colab setup')

In [ ]:
# CELL 2: Setup (run once per session)
import os
import importlib
from pathlib import Path
import _api
import _analysis

# Pick up freshly pulled files in long-lived notebook runtimes.
importlib.reload(_api)
importlib.reload(_analysis)

GAME_NAME = 'Chumpanda'
TAG_LINE  = 'NA1'
REGIONAL  = 'americas'
PLATFORM  = 'na1'

if IN_COLAB:
    from google.colab import userdata
    API_KEY = userdata.get('RIOT_API_KEY')
else:
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.getenv('RIOT_API_KEY', '')

if not API_KEY:
    source = 'Colab secret RIOT_API_KEY' if IN_COLAB else '.env RIOT_API_KEY'
    raise RuntimeError(f'{source} was not found. Add/renew the Riot API key, then rerun Cell 2.')

try:
    _api.init(API_KEY, REGIONAL, PLATFORM)
except TypeError:
    _api.init(API_KEY, REGIONAL)
    if hasattr(_api, '_PLATFORM'):
        _api._PLATFORM = PLATFORM

PUUID = _api.lookup_puuid(GAME_NAME, TAG_LINE)
print('Ready. PUUID:', PUUID[:20], '...')

In [ ]:
# CELL 3: Recent games
from datetime import datetime, timezone

match_ids = _api.get_match_list(PUUID, count=10)

print(f"{'#':>2}  {'Date':>10}  {'Champion':<14}  {'Role':<8}  {'Result':<6}  {'K/D/A':<10}  {'CS/min':>6}  {'Duration'}")
print('-' * 80)

for i, mid in enumerate(match_ids):
    m = _api.get_match(mid)
    info = m['info']
    p = next((x for x in info['participants'] if x['puuid'] == PUUID), None)
    if p is None:
        p = next((x for x in info['participants']
                  if x.get('riotIdGameName', '').lower() == GAME_NAME.lower()), None)
    if p is None:
        print(f"{i:>2}  {mid}  (player not found)")
        continue
    dur = info['gameDuration']
    date = datetime.fromtimestamp(info['gameStartTimestamp'] / 1000, tz=timezone.utc).strftime('%Y-%m-%d')
    cs = p['totalMinionsKilled'] + p.get('neutralMinionsKilled', 0)
    cs_pm = f"{cs * 60 / dur:.1f}" if dur else '-'
    kda = f"{p['kills']}/{p['deaths']}/{p['assists']}"
    result = 'Win' if p['win'] else 'Loss'
    role = p.get('teamPosition') or p.get('individualPosition') or '-'
    duration = f"{dur // 60}m{dur % 60:02d}s"
    print(f"{i:>2}  {date}  {p['championName']:<14}  {role:<8}  {result:<6}  {kda:<10}  {cs_pm:>6}  {duration}")

In [ ]:
# CELL 4: Select a match
# Use the index from the table above (0 = most recent), or paste a specific match ID directly.

MATCH_INDEX = 0
# MATCH_ID = 'NA1_1234567890'  # uncomment to override

MATCH_ID = match_ids[MATCH_INDEX]
print('Selected:', MATCH_ID)

In [ ]:
# CELL 5: Generate coaching file
match_data    = _api.get_match(MATCH_ID)
timeline_data = _api.get_timeline(MATCH_ID)

opponent_info = {}
opponent = _analysis.find_lane_opponent(match_data, PUUID, GAME_NAME, TAG_LINE)
if opponent:
    try:
        opponent_info['champion_mastery'] = _api.get_champion_mastery(opponent['puuid'], opponent['championId'])
    except Exception as exc:
        print('Opponent mastery lookup skipped:', exc)

OUTPUT_MODE = 'compact'  # compact, full_debug, or json
report = _analysis.build_coaching_report(match_data, timeline_data, PUUID, GAME_NAME, TAG_LINE, opponent_info, OUTPUT_MODE)

suffix = 'json' if OUTPUT_MODE == 'json' else 'txt'
out = Path(f"{MATCH_ID}_coaching.{suffix}")
out.write_text(report, encoding='utf-8')
print(f"Saved: {out}")

if IN_COLAB:
    from google.colab import files
    files.download(str(out))

print()
print(report[:3000])